# LHCO initialise-then-optimise experiment (GPU)

Runs `qmaml_hep.experiments.lhco_init` on a Colab GPU. Each run starts the same 92-weight classifier from one of
four points (Learner output, nearest training mass, shared classifier, fresh) and counts the Adam steps needed to
reach 90% of the per-point oracle rejection.

**Before running**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. Upload two files to a Drive folder `MyDrive/qmaml_init/`:
   * `qmaml_init_bundle.tar.gz` (code, selected learning rates, oracle, committed zero-shot reports, ~150 kB)
   * `lhco_sr_features.npz` (signal-region features, ~300 MB, from `qmaml-hep/data/lhco/`)
3. *Runtime → Run all*.

Finished runs are written straight to Drive. If the session disconnects, run all cells again: completed runs are
skipped.

**Note on reproducibility.** GPU arithmetic differs from any CPU's, so the source models' zero-shot numbers will not
be bitwise equal to the committed CPU reports. Each result file records the difference (`reproduction_check`) and
the device. The comparison between starting points is within one platform, so it is unaffected.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing
from google.colab import drive
drive.mount('/content/drive')
print('CPUs:', multiprocessing.cpu_count())

In [ ]:
# ---- settings ----
ROOT = '/content/drive/MyDrive/qmaml_init'      # holds the two uploaded files; results go to ROOT/results
RUN_TUNE = False     # False: reuse the learning rates already tuned on CPU (in the bundle). True: re-tune on this GPU first (+12 short runs).
N_PARALLEL = max(2, multiprocessing.cpu_count())   # concurrent runs on the one GPU; each run needs about one CPU core

In [ ]:
import os, shutil, subprocess, tarfile
CODE = '/content/qmaml'
RESULTS = os.path.join(ROOT, 'results')
LOGS = os.path.join(ROOT, 'logs')
for f in ['qmaml_init_bundle.tar.gz', 'lhco_sr_features.npz']:
    assert os.path.isfile(os.path.join(ROOT, f)), f'missing {ROOT}/{f}'
shutil.rmtree(CODE, ignore_errors=True)
with tarfile.open(os.path.join(ROOT, 'qmaml_init_bundle.tar.gz')) as t:
    t.extractall(CODE)
# seed the Drive results folder with the bundle's inputs (never overwrites finished runs)
for sub in ['lhco_routes', 'lhco_init']:
    src, dst = os.path.join(CODE, 'results', sub), os.path.join(RESULTS, sub)
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        if RUN_TUNE and sub == 'lhco_init' and f.startswith(('tune_', 'selected')):
            continue
        if not os.path.exists(os.path.join(dst, f)):
            shutil.copy(os.path.join(src, f), dst)
os.makedirs(LOGS, exist_ok=True)
# data on local disk (faster than Drive)
os.makedirs('/content/data/lhco', exist_ok=True)
if not os.path.isfile('/content/data/lhco/lhco_sr_features.npz'):
    shutil.copy(os.path.join(ROOT, 'lhco_sr_features.npz'), '/content/data/lhco/')
ENV = dict(os.environ, PYTHONPATH=os.path.join(CODE, 'src'), QMAML_DEVICE='cuda', QMAML_RESULTS=RESULTS,
           QMAML_DATA='/content/data', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
os.environ.update(ENV)
print('ready')

Smoke test: the GPU path runs, agrees with the CPU path, and is fast.

In [ ]:
%%writefile /content/smoke.py
import time, numpy as np, torch, torch.nn as nn, os
from qmaml_hep.experiments import lhco_routes as L
assert L.DEVICE.type == 'cuda'
d = L.Data(); tr, va, te = L.grid_split(d.masses, 100); m = te[0]
model = L.Arm('qmaml', 0)
rng = np.random.default_rng(0); opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], 1e-3)
torch.cuda.synchronize(); t = time.time()
for _ in range(50):
    x, y = d.batch(m, L.EV_PER_TASK, rng)
    loss = nn.functional.cross_entropy(model(x, m), y); opt.zero_grad(); loss.backward(); opt.step()
torch.cuda.synchronize(); print(f'adaptation step {1e3 * (time.time() - t) / 50:.1f} ms')
t = time.time(); r_gpu = L.evaluate(model, d, m, d.q_test)[2]; print(f'evaluation {time.time() - t:.2f} s, R = {r_gpu:.4f}')
# same model on the CPU
x = d.norm(d.sig[m][1][:2000])
with torch.no_grad():
    o_gpu = model(x, m).cpu().double()
    L.DEVICE = torch.device('cpu'); cpu_model = model.to('cpu')
    o_cpu = cpu_model(x.cpu(), m).double()
print(f'max |GPU - CPU| logit difference: {(o_gpu - o_cpu).abs().max().item():.1e}')

In [ ]:
!cd /content && python smoke.py

In [ ]:
import json, time, glob
from concurrent.futures import ThreadPoolExecutor
PY = f"python -m qmaml_hep.experiments.lhco_init"
OUT = os.path.join(RESULTS, 'lhco_init')

def run_all(stage):
    cmds = subprocess.run(f'{PY} jobs --stage {stage}', shell=True, env=ENV, capture_output=True, text=True,
                          cwd='/content', check=True).stdout.split('\n')
    cmds = [c for c in cmds if c]
    todo = []
    for c in cmds:
        a = c.split()
        start, lr, seed = a[a.index('--start') + 1], a[a.index('--lr') + 1], a[a.index('--seed') + 1]
        name = f'{stage}_{start}_lr{lr}_s{seed}'
        if not os.path.isfile(os.path.join(OUT, name + '.json')):
            todo.append((name, c))
    print(f'{stage}: {len(cmds) - len(todo)} done, {len(todo)} to run, {N_PARALLEL} at a time')
    t0 = time.time()
    def one(job):
        name, c = job
        with open(os.path.join(LOGS, name + '.log'), 'w') as log:
            rc = subprocess.run(c, shell=True, env=ENV, cwd='/content', stdout=log, stderr=subprocess.STDOUT).returncode
        print(f'  {name}: {"ok" if rc == 0 else "FAILED rc=" + str(rc)}  ({(time.time() - t0) / 60:.0f} min)', flush=True)
        return rc
    with ThreadPoolExecutor(N_PARALLEL) as ex:
        rcs = list(ex.map(one, todo))
    if any(rcs):
        raise RuntimeError(f'{sum(r != 0 for r in rcs)} runs failed; see {LOGS}')

if RUN_TUNE:
    run_all('tune')
    subprocess.run(f'{PY} select', shell=True, env=ENV, cwd='/content', check=True)
print(open(os.path.join(OUT, 'selected_lr.json')).read())
run_all('report')

In [ ]:
!cd /content && python -m qmaml_hep.experiments.lhco_init score
import json, glob
for f in sorted(glob.glob(os.path.join(OUT, 'report_*.json')))[:1]:
    r = json.load(open(f)); print(r['environment'], r['reproduction_check'], f"{r['seconds'] / 60:.0f} min")

Download everything (or copy `MyDrive/qmaml_init/results/lhco_init` to `qmaml-hep/results/lhco_init_gpu/`).

In [ ]:
shutil.make_archive('/content/lhco_init_gpu', 'zip', OUT)
from google.colab import files
files.download('/content/lhco_init_gpu.zip')